# Training Pipeline
[run_training_dpo_pipeline.ipynb](https://github.com/shibing624/MedicalGPT/blob/main/run_training_dpo_pipeline.ipynb)    | [Open In Colab](https://colab.research.google.com/github/shibing624/MedicalGPT/blob/main/run_training_dpo_pipeline.ipynb)

# Stage 1: Continue Pretraining

第一阶段：PT(Continue PreTraining)增量预训练，在海量领域文本数据上二次预训练GPT模型，以适配领域数据分布

注意：
1. 此阶段是可选的，如果你没有海量领域文本，可以跳过此阶段，直接进行SFT阶段的有监督微调
2. 我实验发现：做领域知识注入，SFT比PT更高效，也可以跳过PT阶段

| Stage 1: Continue Pretraining   |  [pretraining.py](https://github.com/shibing624/MedicalGPT/blob/main/pretraining.py) | [run_pt.sh](https://github.com/shibing624/MedicalGPT/blob/main/run_pt.sh)    |

#### 说明：
以下 notebook/colab 代码为了快速验证训练代码可用，我们使用了小size的生成模型和小样本数据集，实际使用时，需要使用更大的模型和数据集，以获得更好的效果。

1. 生成模型：使用的是Qwen/Qwen2.5-0.5B
2. 数据集：PT阶段使用的是中文天龙八部小说部分文本和英文书籍部分文本，位于`data/pretrain`文件夹

## 配置运行环境

本地执行可注释以下配置环境的命令，colab执行要打开注释，用于配置环境

colab建议使用T4 GPU训练，设置方式：`代码执行程序 -> 更改运行时类型 -> 运行时类型：Python3，硬件加速器：GPU，GPU类型：T4 -> 保存`

步骤：
1. 下载最新代码到本地
2. 安装依赖包

依赖包如下，保证最新版本：

```
loguru
transformers
sentencepiece
datasets
tensorboard
tqdm
peft
trl
```

In [2]:
!git clone --depth 1 https://github.com/shibing624/MedicalGPT.git
%cd MedicalGPT
%ls
!pip install -r requirements.txt

Cloning into 'MedicalGPT'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 102 (delta 16), reused 88 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (102/102), 8.57 MiB | 12.92 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/MedicalGPT
build_domain_tokenizer.py   README.md
chatpdf.py                  requirements.txt
CITATION.cff                reward_modeling.py
_config.yml                 role_play_data/
CONTRIBUTING.md             run_dpo.sh
convert_dataset.py          run_eval_quantize.sh
data/                       run_grpo.sh
DISCLAIMER                  run_orpo.sh
docs/                       run_ppo.sh
dpo_training.py             run_pt.sh
eval_quantize.py            run_quant.sh
fastapi_server_demo.py      run_rm.sh
gradio_demo.py              run_sft_accelerate.sh
grpo_training.py            run_sft.sh
inference_multigpu_demo.py  run_training_dpo_pipe

## Stage1 咱们开始吧

训练步骤如下：

1. 确认训练集
2. 执行训练脚本

训练脚本的执行逻辑如下：
1. 导入依赖包
2. 设置参数
3. 定义各函数并加载训练集
4. 加载模型和tokenizer
5. 开始训练并评估
6. 查看训练结果

**以下参数可以根据你的GPU实际情况修改，当前参数是根据Colab的T4单卡GPU（16GB显存）配置的**

In [3]:
%ls ./data/pretrain/

en_article_tail500.txt  fever.txt  tianlongbabu.txt


In [4]:
!python pretraining.py \
    --model_name_or_path Qwen/Qwen2.5-0.5B \
    --train_file_dir ./data/pretrain \
    --validation_file_dir ./data/pretrain \
    --per_device_train_batch_size 3 \
    --per_device_eval_batch_size 3 \
    --do_train \
    --do_eval \
    --use_peft True \
    --seed 42 \
    --bf16 \
    --max_train_samples 20000 \
    --max_eval_samples 10 \
    --num_train_epochs 1 \
    --learning_rate 2e-4 \
    --warmup_ratio 0.05 \
    --weight_decay 0.01 \
    --logging_strategy steps \
    --logging_steps 10 \
    --eval_steps 50 \
    --eval_strategy steps \
    --save_steps 50 \
    --save_strategy steps \
    --save_total_limit 3 \
    --gradient_accumulation_steps 1 \
    --preprocessing_num_workers 1 \
    --block_size 128 \
    --output_dir outputs-pt-v1 \
    --overwrite_output_dir \
    --ddp_timeout 30000 \
    --logging_first_step True \
    --target_modules all \
    --lora_rank 8 \
    --lora_alpha 16 \
    --lora_dropout 0.05 \
    --torch_dtype bfloat16 \
    --device_map auto \
    --report_to tensorboard \
    --ddp_find_unused_parameters False \
    --gradient_checkpointing True

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
2026-03-25 04:22:44.985 | INFO     | __main__:main:364 - Model args: ModelArguments(model_name_or_path='Qwen/Qwen2.5-0.5B', tokenizer_name_or_path=None, load_in_8bit=False, load_in_4bit=False, cache_dir=None, model_revision='main', hf_hub_token=None, use_fast_tokenizer=False, torch_dtype='bfloat16', device_map='auto', trust_remote_code=True)
2026-03-25 04:22:44.985 | INFO     | __main__:main:365 - Data args: DataArguments(dataset_name=None, dataset_config_name=None, train_file_dir='./data/pretrain', validation_file_dir='./data/pretrain', max_train_samples=20000, max_eval_samples=10, streaming=False, block_size=128, overwrite_cache=False, validation_split_percentage=1, preprocessing_num_workers=1, keep_linebreaks=True, packing=True)
2026-03-25 04:22:44.985 | INFO     | __main__:main:366 - Training args: Seq2SeqTrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_

In [5]:
%ls -lh outputs-pt-v1

total 28M
-rw-r--r-- 1 root root 1.1K Mar 25 04:34 adapter_config.json
-rw-r--r-- 1 root root  17M Mar 25 04:34 adapter_model.safetensors
-rw-r--r-- 1 root root  471 Mar 25 04:34 all_results.json
-rw-r--r-- 1 root root 2.4K Mar 25 04:34 chat_template.jinja
drwxr-xr-x 2 root root 4.0K Mar 25 04:32 checkpoint-750/
drwxr-xr-x 2 root root 4.0K Mar 25 04:33 checkpoint-800/
drwxr-xr-x 2 root root 4.0K Mar 25 04:34 checkpoint-845/
-rw-r--r-- 1 root root  262 Mar 25 04:34 eval_results.json
-rw-r--r-- 1 root root 5.1K Mar 25 04:34 README.md
drwxr-xr-x 3 root root 4.0K Mar 25 04:23 runs/
-rw-r--r-- 1 root root  694 Mar 25 04:34 tokenizer_config.json
-rw-r--r-- 1 root root  11M Mar 25 04:34 tokenizer.json
-rw-r--r-- 1 root root  21K Mar 25 04:34 trainer_state.json
-rw-r--r-- 1 root root  229 Mar 25 04:34 train_results.json


模型训练结果：
- 使用lora训练模型，则保存的lora权重是`adapter_model.safetensors`, lora配置文件是`adapter_config.json`，合并到base model的方法见`merge_peft_adapter.py`
- 日志保存在`output_dir/runs`目录下，可以使用tensorboard查看，启动tensorboard方式如下：`tensorboard --logdir output_dir/runs --host 0.0.0.0 --port 8009`

lora模型权重合并到base model，合并后的模型保存在`--output_dir`目录下，合并方法如下：

In [6]:
!python merge_peft_adapter.py \
    --base_model Qwen/Qwen2.5-0.5B --lora_model outputs-pt-v1 --output_dir merged-pt/

Namespace(base_model='Qwen/Qwen2.5-0.5B', tokenizer_path=None, lora_model='outputs-pt-v1', resize_emb=False, output_dir='merged-pt/', hf_hub_model_id='', hf_hub_token=None)
Base model: Qwen/Qwen2.5-0.5B
LoRA model: outputs-pt-v1
Loading LoRA for causal language model
Loading weights: 100% 290/290 [00:00<00:00, 876.05it/s]
Merging with merge_and_unload...
Saving to Hugging Face format...
Writing model shards: 100% 1/1 [00:04<00:00,  4.22s/it]
Done! model saved to merged-pt/


In [7]:
%ls -lh merged-pt/

total 954M
-rw-r--r-- 1 root root 2.4K Mar 25 04:35 chat_template.jinja
-rw-r--r-- 1 root root 1.3K Mar 25 04:35 config.json
-rw-r--r-- 1 root root  138 Mar 25 04:35 generation_config.json
-rw-r--r-- 1 root root 943M Mar 25 04:35 model.safetensors
-rw-r--r-- 1 root root  668 Mar 25 04:35 tokenizer_config.json
-rw-r--r-- 1 root root  11M Mar 25 04:35 tokenizer.json


In [8]:
%cat merged-pt/config.json

{
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 24,
  "model_type": "qwen2",
  "num_attention_heads": 14,
  "num_hidden_layers": 24,
  "num_key_value_heads": 2,
  "pad_token_id": n

Stage1 增量预训练完成。

# Stage 2: Supervised FineTuning

第二阶段：SFT(Supervised Fine-tuning)有监督微调，构造指令微调数据集，在预训练模型基础上做指令精调，以对齐指令意图，并注入领域知识

| Stage 2: Supervised Fine-tuning | [supervised_finetuning.py](https://github.com/shibing624/MedicalGPT/blob/main/supervised_finetuning.py) | [run_sft.sh](https://github.com/shibing624/MedicalGPT/blob/main/run_sft.sh)  |

#### 说明：
以下 notebook/colab 代码为了快速验证训练代码可用，我们使用了小size的生成模型和小样本数据集，实际使用时，需要使用更大的模型和数据集，以获得更好的效果。

1. 生成模型：使用的是Qwen/Qwen2.5-0.5B 或者 Stage1得到的预训练模型
2. 数据集：SFT阶段使用的是使用的是Belle的1千条抽样数据，位于`data/finetune`文件夹

## Stage2 咱们开始吧

训练步骤如下：

1. 确认训练集
2. 执行训练脚本

训练脚本的执行逻辑如下：
1. 导入依赖包
2. 设置参数
3. 定义各函数并加载训练集
4. 加载模型和tokenizer
5. 开始训练并评估
6. 查看训练结果

In [9]:
%ls ./data/finetune

medical_sft_1K_format.jsonl  sharegpt_zh_1K_format.jsonl


In [13]:
python -c "import transformers; print(transformers.__version__)"
python /content/MedicalGPT/supervised_finetuning.py --help | grep overwrite_output_dir
grep -n "HfArgumentParser" /content/MedicalGPT/supervised_finetuning.py

SyntaxError: invalid syntax (3306917777.py, line 1)

In [14]:
!python supervised_finetuning.py \
    --model_name_or_path merged-pt \
    --train_file_dir ./data/finetune \
    --validation_file_dir ./data/finetune \
    --per_device_train_batch_size 4 \
    --per_device_eval_batch_size 4 \
    --do_train \
    --do_eval \
    --use_peft True \
    --bf16 \
    --max_train_samples 1000 \
    --max_eval_samples 10 \
    --num_train_epochs 1 \
    --learning_rate 2e-5 \
    --warmup_ratio 0.05 \
    --weight_decay 0.05 \
    --logging_strategy steps \
    --logging_steps 10 \
    --eval_steps 50 \
    --eval_strategy steps \
    --save_steps 500 \
    --save_strategy steps \
    --save_total_limit 3 \
    --gradient_accumulation_steps 1 \
    --preprocessing_num_workers 1 \
    --output_dir outputs-sft-v1 \
    --ddp_timeout 30000 \
    --logging_first_step True \
    --target_modules all \
    --lora_rank 8 \
    --lora_alpha 16 \
    --lora_dropout 0.05 \
    --torch_dtype bfloat16 \
    --device_map auto \
    --report_to tensorboard \
    --ddp_find_unused_parameters False \
    --gradient_checkpointing True

2026-03-25 04:46:39.338 | WARNING  | __main__:__post_init__:192 - You may set max_train_samples = -1 to run all samples in production.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
2026-03-25 04:46:39.576 | INFO     | __main__:main:346 - Model args: ModelArguments(model_name_or_path='merged-pt', load_in_8bit=False, load_in_4bit=False, tokenizer_name_or_path=None, cache_dir=None, model_revision='main', hf_hub_token=None, use_fast_tokenizer=False, torch_dtype='bfloat16', device_map='auto', trust_remote_code=True, rope_scaling=None, flash_attn=False, shift_attn=False, neft_alpha=0)
2026-03-25 04:46:39.576 | INFO     | __main__:main:347 - Data args: DataArguments(dataset_name=None, dataset_config_name=None, train_file_dir='./data/finetune', validation_file_dir='./data/finetune', max_train_samples=1000, max_eval_samples=10, ignore_pad_token_for_loss=True, overwrite_cache=False, validation_split_percentage=1, preprocessing_num_workers=1)
2026-03-25 04:46

In [15]:
%ls -lh outputs-sft-v1

total 28M
-rw-r--r-- 1 root root 1.1K Mar 25 04:59 adapter_config.json
-rw-r--r-- 1 root root  17M Mar 25 04:59 adapter_model.safetensors
-rw-r--r-- 1 root root  426 Mar 25 04:59 all_results.json
-rw-r--r-- 1 root root 2.4K Mar 25 04:59 chat_template.jinja
drwxr-xr-x 2 root root 4.0K Mar 25 04:59 checkpoint-250/
-rw-r--r-- 1 root root  218 Mar 25 04:59 eval_results.json
-rw-r--r-- 1 root root 5.1K Mar 25 04:59 README.md
drwxr-xr-x 3 root root 4.0K Mar 25 04:46 runs/
-rw-r--r-- 1 root root  704 Mar 25 04:59 tokenizer_config.json
-rw-r--r-- 1 root root  11M Mar 25 04:59 tokenizer.json
-rw-r--r-- 1 root root 6.3K Mar 25 04:59 trainer_state.json
-rw-r--r-- 1 root root  228 Mar 25 04:59 train_results.json


模型训练结果：
- 使用lora训练模型，则保存的lora权重是`adapter_model.safetensors`, lora配置文件是`adapter_config.json`，合并到base model的方法见`merge_peft_adapter.py`
- 日志保存在`output_dir/runs`目录下，可以使用tensorboard查看，启动tensorboard方式如下：`tensorboard --logdir output_dir/runs --host 0.0.0.0 --port 8009`

lora模型权重合并到base model，合并后的模型保存在`--output_dir`目录下，合并方法如下：

In [16]:
!python merge_peft_adapter.py \
    --base_model merged-pt --lora_model outputs-sft-v1 --output_dir ./merged-sft

Namespace(base_model='merged-pt', tokenizer_path=None, lora_model='outputs-sft-v1', resize_emb=False, output_dir='./merged-sft', hf_hub_model_id='', hf_hub_token=None)
Base model: merged-pt
LoRA model: outputs-sft-v1
Loading LoRA for causal language model
Loading weights: 100% 290/290 [00:00<00:00, 868.49it/s]
Merging with merge_and_unload...
Saving to Hugging Face format...
Writing model shards: 100% 1/1 [00:06<00:00,  6.42s/it]
Done! model saved to ./merged-sft


In [17]:
%ls -lh merged-sft/

total 954M
-rw-r--r-- 1 root root 2.4K Mar 25 05:02 chat_template.jinja
-rw-r--r-- 1 root root 1.3K Mar 25 05:02 config.json
-rw-r--r-- 1 root root  138 Mar 25 05:02 generation_config.json
-rw-r--r-- 1 root root 943M Mar 25 05:02 model.safetensors
-rw-r--r-- 1 root root  667 Mar 25 05:02 tokenizer_config.json
-rw-r--r-- 1 root root  11M Mar 25 05:02 tokenizer.json


In [18]:
%cat merged-sft/config.json

{
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 24,
  "model_type": "qwen2",
  "num_attention_heads": 14,
  "num_hidden_layers": 24,
  "num_key_value_heads": 2,
  "pad_token_id": n

Stage2 SFT训练完成。

# Stage 3: DPO(Direct Preference Optimization)

第三阶段：DPO(Direct Preference Optimization)直接偏好优化，DPO通过直接优化语言模型来实现对其行为的精确控制，而无需使用复杂的强化学习，也可以有效学习到人类偏好，DPO相较于RLHF更容易实现且易于训练，效果更好

| Stage 3: Direct Preference Optimization        |  [dpo_training.py](https://github.com/shibing624/MedicalGPT/blob/main/dpo_training.py) | [run_dpo.sh](https://github.com/shibing624/MedicalGPT/blob/main/run_dpo.sh)    |

#### 说明：
以下 notebook/colab 代码为了快速验证训练代码可用，我们使用了小size的生成模型和小样本数据集，实际使用时，需要使用更大的模型和数据集，以获得更好的效果。

1. 生成模型：使用的是`Qwen/Qwen2.5-0.5B` 或者 Stage2得到的SFT模型
2. 数据集：DPO阶段使用的是医疗reward数据，抽样了500条，位于`data/reward`文件夹

## Stage3 咱们开始吧

训练步骤如下：

1. 确认训练集
2. 执行训练脚本

训练脚本的执行逻辑如下：
1. 导入依赖包
2. 设置参数
3. 定义各函数并加载训练集
4. 加载模型和tokenizer
5. 开始训练并评估
6. 查看训练结果

In [19]:
%ls ./data/reward/

dpo_zh_500.jsonl


In [20]:
!python dpo_training.py \
    --model_name_or_path ./merged-sft \
    --template_name qwen \
    --train_file_dir ./data/reward \
    --validation_file_dir ./data/reward \
    --per_device_train_batch_size 3 \
    --per_device_eval_batch_size 1 \
    --do_train \
    --do_eval \
    --use_peft True \
    --max_train_samples 1000 \
    --max_eval_samples 500 \
    --max_steps 100 \
    --eval_steps 10 \
    --save_steps 50 \
    --max_source_length 256 \
    --max_target_length 256 \
    --output_dir outputs-dpo-v1 \
    --target_modules all \
    --lora_rank 8 \
    --lora_alpha 16 \
    --lora_dropout 0.05 \
    --torch_dtype bfloat16 \
    --bf16 True \
    --fp16 False \
    --device_map auto \
    --report_to tensorboard \
    --remove_unused_columns False \
    --gradient_checkpointing True \
    --cache_dir ./cache

2026-03-25 05:03:25.823 | INFO     | __main__:main:198 - Parse args: ScriptArguments(model_name_or_path='./merged-sft', tokenizer_name_or_path=None, load_in_8bit=False, load_in_4bit=False, cache_dir='./cache', use_fast_tokenizer=False, torch_dtype='bfloat16', device_map='auto', trust_remote_code=True, dataset_name=None, dataset_config_name=None, train_file_dir='./data/reward', validation_file_dir='./data/reward', template_name='qwen', per_device_train_batch_size=3, per_device_eval_batch_size=1, max_source_length=256, max_target_length=256, min_target_length=4, max_train_samples=1000, max_eval_samples=500, overwrite_cache=False, validation_split_percentage=1, preprocessing_num_workers=4, use_peft=True, qlora=False, target_modules='all', lora_rank=8, lora_dropout=0.05, lora_alpha=16.0, peft_path=None, do_train=True, do_eval=True, learning_rate=0.0005, lr_scheduler_type='cosine', warmup_steps=100, weight_decay=0.05, optim='adamw_torch', fp16=False, bf16=True, gradient_checkpointing=True, 

In [21]:
%ls -lh outputs-dpo-v1

total 28M
-rw-r--r-- 1 root root 1.1K Mar 25 05:34 adapter_config.json
-rw-r--r-- 1 root root  17M Mar 25 05:34 adapter_model.safetensors
-rw-r--r-- 1 root root  380 Mar 25 05:35 all_results.json
-rw-r--r-- 1 root root 2.4K Mar 25 05:34 chat_template.jinja
drwxr-xr-x 2 root root 4.0K Mar 25 05:34 checkpoint-100/
drwxr-xr-x 2 root root 4.0K Mar 25 05:18 checkpoint-50/
-rw-r--r-- 1 root root  170 Mar 25 05:35 eval_results.json
-rw-r--r-- 1 root root 2.3K Mar 25 05:34 README.md
drwxr-xr-x 3 root root 4.0K Mar 25 05:03 runs/
-rw-r--r-- 1 root root  678 Mar 25 05:34 tokenizer_config.json
-rw-r--r-- 1 root root  11M Mar 25 05:34 tokenizer.json
-rw-r--r-- 1 root root  71K Mar 25 05:34 trainer_state.json
-rw-r--r-- 1 root root 5.7K Mar 25 05:34 training_args.bin
-rw-r--r-- 1 root root  212 Mar 25 05:34 train_results.json


模型训练结果：
- 使用lora训练模型，则保存的lora权重是`adapter_model.safetensors`, lora配置文件是`adapter_config.json`，合并到base model的方法见`merge_peft_adapter.py`
- 日志保存在`output_dir/runs`目录下，可以使用tensorboard查看，启动tensorboard方式如下：`tensorboard --logdir output_dir/runs --host 0.0.0.0 --port 8009`

lora模型权重合并到base model，合并后的模型保存在`--output_dir`目录下，合并方法如下：

In [23]:
!python merge_peft_adapter.py \
    --base_model merged-sft --lora_model outputs-dpo-v1 --output_dir merged-dpo/

Namespace(base_model='merged-sft', tokenizer_path=None, lora_model='outputs-dpo-v1', resize_emb=False, output_dir='merged-dpo/', hf_hub_model_id='', hf_hub_token=None)
Base model: merged-sft
LoRA model: outputs-dpo-v1
Loading LoRA for causal language model
Loading weights: 100% 290/290 [00:00<00:00, 910.20it/s]
Merging with merge_and_unload...
Saving to Hugging Face format...
Writing model shards: 100% 1/1 [00:06<00:00,  6.16s/it]
Done! model saved to merged-dpo/


In [24]:
%ls -lh merged-dpo/

total 954M
-rw-r--r-- 1 root root 2.4K Mar 25 05:52 chat_template.jinja
-rw-r--r-- 1 root root 1.3K Mar 25 05:52 config.json
-rw-r--r-- 1 root root  138 Mar 25 05:52 generation_config.json
-rw-r--r-- 1 root root 943M Mar 25 05:52 model.safetensors
-rw-r--r-- 1 root root  667 Mar 25 05:52 tokenizer_config.json
-rw-r--r-- 1 root root  11M Mar 25 05:52 tokenizer.json


In [25]:
%cat merged-dpo/config.json

{
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 24,
  "model_type": "qwen2",
  "num_attention_heads": 14,
  "num_hidden_layers": 24,
  "num_key_value_heads": 2,
  "pad_token_id": n

Stage3 偏好建模第一次训练完成。

**至此一个完整的训练流程演示完成。**

# Test

In [27]:
!python inference.py --base_model merged-dpo
# 或在shell中运行
# python inference.py --base_model merged-dpo --interactive

Namespace(base_model='merged-dpo', lora_model='', tokenizer_path=None, system_prompt='', stop_str='', repetition_penalty=1.0, max_new_tokens=512, data_file=None, interactive=False, single_tune=False, temperature=0.7, output_file='./predictions_result.jsonl', eval_batch_size=4, resize_emb=False, load_in_8bit=False, load_in_4bit=False)
Loading weights: 100% 290/290 [00:00<00:00, 913.32it/s]
Qwen2Tokenizer(name_or_path='merged-dpo', vocab_size=151643, model_max_length=131072, padding_side='left', truncation_side='right', special_tokens={'eos_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object

Input:介绍下南京
Response:  南京市位于江苏省西南部，是全国首批历史文化名城、国家中心城市和自由贸易试验区。

完。
